In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import tqdm

import torchvision.utils
from torchvision.io import read_image, ImageReadMode

import torch.nn.functional as F

import cv2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from ultralytics import YOLO

import os

from PIL import Image

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/home/atin/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
yolo_model = YOLO('yolo11m-pose.pt')
# yolo_model.to(device)
def extract_pose_features(video_path, max_frames=20, max_objects=64, confidence_threshold=0.2, iou_threshold=0.7):
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    
    # print(f"Video info: {total_frames} frames, {fps} FPS, {duration:.2f} seconds")
    
    if total_frames > max_frames:
        frame_indices = np.linspace(0, total_frames-1, max_frames, dtype=int)
    else:
        frame_indices = range(min(total_frames, max_frames))
    
    pose_data = []
    
    frame_count = 0
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        
        ret, frame = cap.read()
        if not ret:
            # print(f"Warning: Could not read frame {frame_idx}")
            pose_data.append([[0] * (17*2) for _ in range(max_objects)])
            # pbar.update(1)
            continue
        
        results = yolo_model(frame, verbose=False, conf=confidence_threshold, iou=iou_threshold, max_det=max_objects, device="cuda:0")
        
        frame_poses = []
        
        if len(results) > 0 and hasattr(results[0], 'keypoints') and results[0].keypoints is not None:
            keypoints = results[0].keypoints.data.cpu().numpy()
            # Normalize by dividing by image dimensions
            keypoints[..., 0] /= frame.shape[1]  # normalize x by width
            keypoints[..., 1] /= frame.shape[0]  # normalize y by height
            
            valid_keypoints = keypoints[..., :-1]

            for i in range(min(len(valid_keypoints), max_objects)):
                person_keypoints = valid_keypoints[i].flatten().tolist()
                frame_poses.append(person_keypoints)
       
        l = len(frame_poses) 
        while len(frame_poses) < max_objects:
            cur = len(frame_poses)
            if l == 0:
                frame_poses.append([0] * (17*2))  # 17 keypoints with x,y
            else:
                frame_poses.append(frame_poses[cur % l])  # 17 keypoints with x,y,conf
        
        # If we have more than max_objects, truncate
        frame_poses = frame_poses[:max_objects]
        
        # Add to results
        pose_data.append(frame_poses)
        
        frame_count += 1
    
    cap.release()
    
    while len(pose_data) < max_frames:
        pose_data.append([[0] * (17*2) for _ in range(max_objects)])
    
    pose_data = pose_data[:max_frames]
    
    # print(f"Processed {frame_count} frames, extracted pose data with shape: {len(pose_data)} frames, {max_objects} objects, 17 keypoints")
    
    pose_data = np.array(pose_data, dtype=np.float32)
    # print(pose_data.shape)
    return pose_data

In [4]:
class PoseFeatureCNN(nn.Module):
    def __init__(self, num_objects=64, num_keypoints=17, output_dim=1024):
        super(PoseFeatureCNN, self).__init__()
        
        input_features = num_objects * num_keypoints * 2
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_features, 4096),
            nn.ReLU(),
            nn.Linear(4096, 2048),
            nn.ReLU(),
            nn.Linear(2048, output_dim)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        
        x = x.view(batch_size, -1)
        
        x = self.feature_extractor(x)
        
        return x

In [ ]:
class PoseCNNLSTM(nn.Module):
    """
    Combined CNN-LSTM model for pose-based violence detection.
    The CNN extracts features from each frame's pose data,
    then the LSTM processes the sequence of features over time.
    
    Input: (batch_size, frames, max_objects, keypoints*3)
    Output: (batch_size, 1) - violence probability
    """
    def __init__(self, input_frames=20, max_objects=64, num_keypoints=17, 
                 cnn_output_dim=1024, lstm_hidden_dim=512, num_layers=2, dropout=0.2):
        super(PoseCNNLSTM, self).__init__()
        
        # CNN feature extractor
        self.cnn = PoseFeatureCNN()
        
        # Save parameters
        self.input_frames = input_frames
        self.max_objects = max_objects
        self.num_keypoints = num_keypoints
        
        # LSTM for temporal reasoning
        self.lstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        
        # Attention layer (optional but helpful)
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, 1),  # *2 for bidirectional
            nn.Tanh()
        )
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, 512),  # *2 for bidirectional
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),  # Output single probability
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),  # Final output layer
            nn.Sigmoid()
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Forward pass through the network
        
        Args:
            x: Input tensor of shape (batch_size, frames, max_objects, keypoints*3)
            
        Returns:
            output: Violence probability (batch_size, 1)
        """
        
        batch_size = x.size(0)
        x = x.view(batch_size * self.input_frames, self.max_objects, self.num_keypoints * 2)

        features = self.cnn(x)  # (batch_size * frames, cnn_output_dim)
        print(features)
        sequence = features.view(batch_size, self.input_frames, -1)
        
        # Process through LSTM
        lstm_out, (h_n, c_n) = self.lstm(sequence)
        # lstm_out shape: (batch_size, frames, lstm_hidden_dim*2)
        
        # # Attention mechanism (optional)
        # attention_scores = self.attention(lstm_out).squeeze(-1)  # (batch_size, frames)
        # attention_weights = F.softmax(attention_scores, dim=1).unsqueeze(2)  # (batch_size, frames, 1)
        
        # # Apply attention weights
        # context_vector = torch.sum(lstm_out * attention_weights, dim=1)  # (batch_size, lstm_hidden_dim*2)
        
        # Final classification
        output = self.classifier(lstm_out[:, -1, :])  # Use last frame's output
        # output = self.classifier(context_vector)  # If using attention
        
        return output


In [6]:
model = PoseCNNLSTM()

model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.003)
criterion = nn.BCELoss()

In [7]:
class ViolenceDataset(Dataset):
    def __init__(self, dir):
        self.dir = dir

        self.classes = ['violence', 'nonviolence']
        self.videos = []
        for cls in self.classes:
            cls_dir = os.path.join(self.dir, cls)
            for video in os.listdir(cls_dir):
                self.videos.append((os.path.join(cls, video), cls == 'violence'))

        self.num_frames = 20
    
    def __len__(self):
        return len(self.videos)
    
    def __getitem__(self, idx):
        vid_path, label = self.videos[idx]
        vid_path = os.path.join(self.dir, vid_path)
        
        pose_features = extract_pose_features(vid_path, max_frames=self.num_frames)
        # print(pose_features.shape)
        return pose_features, float(label)
    
dataset = ViolenceDataset('data/hockeyfight')

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [8]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import seaborn as sns

history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': []
}

total_epoch = 10
for epoch in range(total_epoch):
    model.train()
    train_loss = 0.0
    
    for i, (frames, labels) in enumerate(tqdm.tqdm(train_loader, desc=f'Training Epoch {epoch+1}')):
        if i >= 20: break
        frames = frames.to(device)
        labels = labels.to(device).float()
        
        optimizer.zero_grad()
        # print(frames.shape)
        outputs = model(frames.float())
        print(outputs.squeeze(-1), labels)
        loss = criterion(outputs.squeeze(-1), labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss/len(train_loader)
    history['train_loss'].append(avg_train_loss)
    
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    for i, (frames, labels) in enumerate(tqdm.tqdm(val_loader, desc='Validating')):
        if i >= 5: break
        frames = frames.to(device)
        labels = labels.to(device).float()
        
        with torch.no_grad():
            outputs = model(frames.float())
            outputs = outputs.squeeze(-1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            pred = (outputs > 0.5).float()
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = val_loss/len(val_loader)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    accuracy = sum([1 for p, l in zip(all_preds, all_labels) if p == l]) / len(all_labels)
    
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(accuracy)
    history['val_precision'].append(precision)
    history['val_recall'].append(recall)
    history['val_f1'].append(f1)
    
    print(f'\nEpoch [{epoch+1}/{total_epoch}]')
    print(f'Training Loss: {avg_train_loss:.4f}')
    print(f'Validation Loss: {avg_val_loss:.4f}')
    print(f'Validation Metrics:')
    print(f'  Accuracy: {accuracy*100:.2f}%')
    print(f'  Precision: {precision*100:.2f}%')
    print(f'  Recall: {recall*100:.2f}%')
    print(f'  F1-Score: {f1*100:.2f}%')
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Epoch {epoch+1}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_accuracy'], label='Accuracy')
plt.plot(history['val_precision'], label='Precision')
plt.plot(history['val_recall'], label='Recall')
plt.plot(history['val_f1'], label='F1-Score')
plt.title('Validation Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()

plt.tight_layout()
plt.show()

Training Epoch 1:   2%|▏         | 1/50 [00:08<07:16,  8.91s/it]

tensor([[ 0.0029, -0.0591, -0.0342,  ...,  0.0189, -0.0183, -0.0153],
        [ 0.0117, -0.0391, -0.0023,  ..., -0.0049, -0.0235, -0.0236],
        [ 0.0220, -0.0343, -0.0293,  ..., -0.0254, -0.0280, -0.0187],
        ...,
        [-0.0167, -0.0431,  0.0153,  ...,  0.0193, -0.0019,  0.0346],
        [ 0.0145, -0.0607, -0.0426,  ...,  0.0095, -0.0400, -0.0101],
        [-0.0460, -0.0652, -0.0397,  ...,  0.0353,  0.0182,  0.0146]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169, 0.5169], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor([1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0., 1., 1.], device='cuda:0')


Training Epoch 1:   4%|▍         | 2/50 [00:16<06:38,  8.29s/it]

tensor([[-12.9948, -12.6844,  15.4567,  ..., -17.9162,  13.8765,  13.6627],
        [-13.6419, -13.3601,  16.2357,  ..., -18.8159,  14.5701,  14.3878],
        [-13.2407, -13.0236,  15.8593,  ..., -18.2827,  14.1660,  14.0275],
        ...,
        [-13.6628, -13.3964,  16.3391,  ..., -18.8564,  14.5885,  14.4887],
        [-13.7559, -13.4529,  16.4305,  ..., -18.9494,  14.6952,  14.5773],
        [-13.8513, -13.5228,  16.5189,  ..., -19.0870,  14.8239,  14.6234]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([0.5346, 0.5345, 0.5346, 0.5345, 0.5347, 0.5345, 0.5345, 0.5344, 0.5345, 0.5346, 0.5345, 0.5340, 0.5344, 0.5344, 0.5345, 0.5345], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor([1., 1., 0., 1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1.], device='cuda:0')


Training Epoch 1:   6%|▌         | 3/50 [00:24<06:23,  8.15s/it]

tensor([[  5.9402, -75.0801,   1.4876,  ..., -81.0492,  71.3647,  67.2528],
        [  5.4843, -68.5781,   1.3237,  ..., -73.9493,  65.1658,  61.4005],
        [  5.4297, -69.4078,   1.4185,  ..., -75.0033,  66.0402,  62.1732],
        ...,
        [  4.8871, -62.3382,   1.2491,  ..., -67.3192,  59.2997,  55.8344],
        [  5.2241, -66.9286,   1.3330,  ..., -72.2897,  63.6455,  59.9430],
        [  5.5202, -70.1287,   1.3978,  ..., -75.7093,  66.7049,  62.8137]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([0.6938, 0.6938, 0.6938, 0.6938, 0.6947, 0.6945, 0.6938, 0.6938, 0.6938, 0.6947, 0.6938, 0.6938, 0.6939, 0.6940, 0.6938, 0.6938], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor([0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0.], device='cuda:0')


Training Epoch 1:   8%|▊         | 4/50 [00:32<06:15,  8.17s/it]

tensor([[ -29.8464, -103.8537,    1.7109,  ..., -105.0298,   47.7020,   32.6894],
        [ -28.0923,  -97.8035,    1.6386,  ...,  -98.9425,   44.9663,   30.7939],
        [ -28.3426,  -98.6233,    1.6566,  ...,  -99.7580,   45.3109,   31.0615],
        ...,
        [ -24.9332,  -86.8727,    1.4528,  ...,  -87.8547,   39.8851,   27.3114],
        [ -21.1677,  -73.6858,    1.2430,  ...,  -74.5350,   33.8662,   23.1866],
        [ -28.1301,  -98.0884,    1.6330,  ...,  -99.2015,   45.0396,   30.8654]], device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5374, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373, 0.5373], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 1., 0., 1., 0.], device='cuda:0')


Training Epoch 1:   8%|▊         | 4/50 [00:38<07:27,  9.73s/it]


KeyboardInterrupt: 